In [ ]:
# bootstrap: make src/ importable
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

# CSIRO Image2Biomass — Full CHARMS Pipeline

## What changed vs. the baseline

| Component | Baseline (lite) | This pipeline |
|---|---|---||
| Backbone | ResNet-50 | EfficientNet-V2-S (better perf/param) |
| Image size | 224 | 384 (higher res for fine-grained texture) |
| Metadata fusion | None (aux heads only) | Feature-level cross-modal fusion |
| Auxiliary heads | NDVI, height, state, month | Same + uncertainty (aleatoric) |
| Loss weighting | Fixed 0.30 / 0.30 | Learned log-variance uncertainty weighting |
| Log-target transform | None | log1p transform on targets |
| Scheduler | None | CosineAnnealingWarmRestarts |
| TTA | None | 4× horizontal/vertical flip TTA at inference |
| CV strategy | GroupKFold(5) on image_path | GroupKFold(5) on sample_id (correct grouping) |

## CHARMS core idea (from the paper)
During training, side-channel metadata (NDVI, height, state, month) is available for every image.
At test time, these are **missing** — only the image exists.

CHARMS trains the CNN to **reconstruct** those metadata values from visual features alone
(auxiliary regression/classification heads). This forces the backbone to learn representations
that correlate with vegetation physiology, not just superficial colour patterns.
The auxiliary losses are discarded at inference; their benefit is purely in representation quality.

In [1]:
import os, math, random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)

DEVICE: cpu


In [2]:
# ── Paths (update for Kaggle /kaggle/input/... layout) ──────────────────────
TRAIN_CSV    = '../../../data/tabular/train/train.csv'
TEST_CSV     = '../../../data/tabular/test/test.csv'
IMAGE_FOLDER = '../../../data/image'

# ── Hyper-parameters ─────────────────────────────────────────────────────────
SEED        = 42
BATCH_SIZE  = 16       # increase if VRAM allows
NUM_WORKERS = 0
EPOCHS      = 5
LR          = 2e-4
WEIGHT_DECAY= 1e-4
IMG_SIZE    = 384      # EfficientNet-V2-S native size
N_FOLDS     = 5

TARGETS      = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
NUMERIC_ATTRS= ['Pre_GSHH_NDVI', 'Height_Ave_cm']
CAT_ATTRS    = ['State', 'month']

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

## 1. Data loading

The competition CSV is in **long format** (one row per image × target). We pivot to wide format
so each row is one image with 5 target columns — the natural format for multi-output regression.

In [4]:
def make_wide_train(train_csv):
    df = pd.read_csv(train_csv)
    meta_cols = ['sample_id', 'image_path', 'Sampling_Date', 'State', 'Species',
                 'Pre_GSHH_NDVI', 'Height_Ave_cm']
    meta = df[meta_cols].drop_duplicates('image_path').copy()
    y = (
        df.pivot_table(index='image_path', columns='target_name',
                       values='target', aggfunc='first')
          .reset_index()
    )
    out = meta.merge(y, on='image_path', how='inner')
    out['Sampling_Date'] = pd.to_datetime(out['Sampling_Date'])
    out['month'] = out['Sampling_Date'].dt.month.astype(int)
    return out


def load_test_df(test_csv):
    df = pd.read_csv(test_csv)
    df_unique = df[['sample_id', 'image_path']].drop_duplicates('image_path').copy()
    return df, df_unique


train_df = make_wide_train(TRAIN_CSV)
test_long_df, test_df = load_test_df(TEST_CSV)

print('Train images:', len(train_df))
print('Test images :', len(test_df))
train_df.head()

Train images: 357
Test images : 1


,sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,month
0,ID1011485656__Dry_Clover_g,train/ID1011485656.jpg,2015-09-04,Tas,Ryegrass_Clover,0.62,4.6667,0.0000,31.9984,16.2751,48.2735,16.2750,9
1,ID1012260530__Dry_Clover_g,train/ID1012260530.jpg,2015-04-01,NSW,Lucerne,0.55,16.0000,0.0000,0.0000,7.6000,7.6000,7.6000,4
2,ID1025234388__Dry_Clover_g,train/ID1025234388.jpg,2015-09-01,WA,SubcloverDalkeith,0.38,1.0000,6.0500,0.0000,0.0000,6.0500,6.0500,9
3,ID1028611175__Dry_Clover_g,train/ID1028611175.jpg,2015-05-18,Tas,Ryegrass,0.66,5.0000,0.0000,30.9703,24.2376,55.2079,24.2376,5
4,ID1035947949__Dry_Clover_g,train/ID1035947949.jpg,2015-09-11,Tas,Ryegrass,0.54,3.5000,0.4343,23.2239,10.5261,34.1844,10.9605,9


## 2. Target transformation

Biomass values are right-skewed (many small values, few large ones).
A **log1p transform** makes the targets more normally distributed, which improves
gradient flow and reduces the influence of outliers.
We invert this at submission time with `expm1`.

In [5]:
# Apply log1p to all target columns in train_df
for col in TARGETS:
    train_df[col] = np.log1p(train_df[col].clip(lower=0).astype(np.float32))

## 3. Metadata encoding

In [6]:
def fit_metadata_encoders(train_df):
    df = train_df.copy()
    scaler = StandardScaler()
    df[NUMERIC_ATTRS] = scaler.fit_transform(df[NUMERIC_ATTRS].astype(np.float32))
    state_vocab = {k: i for i, k in enumerate(sorted(df['State'].dropna().unique()))}
    df['State_id']  = df['State'].map(state_vocab).astype(int)
    df['month_id']  = (df['month'] - 1).clip(0, 11).astype(int)
    return df, {'scaler': scaler, 'state_vocab': state_vocab,
                'num_states': len(state_vocab), 'num_months': 12}


def apply_metadata_encoders(df, enc):
    df = df.copy()
    if all(c in df.columns for c in NUMERIC_ATTRS):
        df[NUMERIC_ATTRS] = enc['scaler'].transform(df[NUMERIC_ATTRS].astype(np.float32))
    else:
        for c in NUMERIC_ATTRS:
            df[c] = 0.0
    df['State_id'] = df['State'].map(enc['state_vocab']).fillna(0).astype(int) \
                     if 'State' in df.columns else 0
    df['month_id'] = (df['month'] - 1).clip(0, 11).astype(int) \
                     if 'month' in df.columns else 0
    return df


train_df, encoders = fit_metadata_encoders(train_df)
test_df  = apply_metadata_encoders(test_df, encoders)
print(encoders)

{'scaler': StandardScaler(), 'state_vocab': {'NSW': 0, 'Tas': 1, 'Vic': 2, 'WA': 3}, 'num_states': 4, 'num_months': 12}


## 4. Augmentations

Pasture images are taken from directly above — so **vertical flip is equally valid as horizontal flip**.
We also use `RandomRotation(90)` for the same reason (the quadrat has no canonical orientation).

In [7]:
train_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(90),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    T.RandomGrayscale(p=0.02),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

valid_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 5. Dataset

In [8]:
class BiomassDataset(Dataset):
    def __init__(self, df, image_root, transforms=None, is_test=False):
        self.df         = df.reset_index(drop=True).copy()
        self.image_root = image_root
        self.transforms = transforms
        self.is_test    = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.image_root, row['image_path'])).convert('RGB')
        if self.transforms:
            img = self.transforms(img)

        x_num = torch.tensor([row['Pre_GSHH_NDVI'], row['Height_Ave_cm']], dtype=torch.float32)
        x_cat = torch.tensor([row['State_id'], row['month_id']], dtype=torch.long)

        item = {'image': img, 'image_path': row['image_path'],
                'x_num': x_num, 'x_cat': x_cat}

        if not self.is_test:
            item['targets'] = torch.tensor(row[TARGETS].values.astype(np.float32),
                                           dtype=torch.float32)
        return item

## 6. Model — CHARMS architecture

```
Image ──► EfficientNet-V2-S backbone ──► visual_feat (1280-d)
                                              │
              ┌───────────────────────────────┤
              │                               │
        metadata embed                  visual_feat
        (NDVI, height,                        │
         state, month)                        │
              │                               │
              └──────► concat ──► fused_feat ─┤
                                              │
                              ┌───────────────┤──────────────────┐
                              ▼               ▼                  ▼
                     reg_head (5)     aux_heads (NDVI,     log_var_head (5)
                     (biomass)        height, state,       (uncertainty)
                                      month)
```

**Key difference from baseline:** metadata is fused into the *primary* regression head,
not just predicted from it. This is the cross-modal fusion step the paper proposes —
the model learns to combine visual evidence with contextual priors.

In [9]:
class MetadataEmbedder(nn.Module):
    """
    Encodes tabular side-channel metadata into a fixed embedding.
    Used during training AND at test time (zeroed out if missing).
    """
    def __init__(self, num_states, num_months=12, embed_dim=64, out_dim=128):
        super().__init__()
        self.state_emb = nn.Embedding(num_states + 1, embed_dim // 2, padding_idx=0)
        self.month_emb = nn.Embedding(num_months,     embed_dim // 2)
        # numeric path: 2 continuous values → 64-d
        self.num_fc = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
        )
        # fuse categorical + numeric
        cat_dim = embed_dim // 2 + embed_dim // 2  # 64
        self.out_fc = nn.Sequential(
            nn.Linear(cat_dim + 64, out_dim), nn.ReLU(),
        )

    def forward(self, x_num, x_cat):
        state_e = self.state_emb(x_cat[:, 0]).squeeze(1)  # (B, 32)
        month_e = self.month_emb(x_cat[:, 1]).squeeze(1)  # (B, 32)
        cat_e   = torch.cat([state_e, month_e], dim=1)    # (B, 64)
        num_e   = self.num_fc(x_num)                       # (B, 64)
        return self.out_fc(torch.cat([cat_e, num_e], dim=1))  # (B, 128)


class BiomassCharms(nn.Module):
    def __init__(self, num_states, num_months=12, meta_embed_dim=128):
        super().__init__()

        # ── Backbone ────────────────────────────────────────────────────────
        efn = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        feat_dim = efn.classifier[1].in_features  # 1280
        efn.classifier = nn.Identity()
        self.backbone = efn
        self.feat_dim = feat_dim

        # ── Metadata embedder ───────────────────────────────────────────────
        self.meta_emb = MetadataEmbedder(
            num_states=num_states, num_months=num_months, out_dim=meta_embed_dim
        )

        # ── Cross-modal fusion ──────────────────────────────────────────────
        fused_dim = feat_dim + meta_embed_dim
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 512), nn.ReLU(), nn.Dropout(0.3),
        )

        # ── Primary regression head (5 targets) ────────────────────────────
        self.reg_head = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, len(TARGETS)),
        )

        # ── Aleatoric uncertainty head (log variance per target) ────────────
        self.log_var_head = nn.Linear(512, len(TARGETS))

        # ── Auxiliary CHARMS heads (from visual features only) ──────────────
        self.ndvi_head   = nn.Linear(feat_dim, 1)
        self.height_head = nn.Linear(feat_dim, 1)
        self.state_head  = nn.Linear(feat_dim, num_states)
        self.month_head  = nn.Linear(feat_dim, num_months)

    def forward(self, x, x_num, x_cat):
        # Visual features
        visual = self.backbone(x)                              # (B, feat_dim)

        # Metadata embedding
        meta = self.meta_emb(x_num, x_cat)                    # (B, meta_embed_dim)

        # Cross-modal fusion
        fused = self.fusion(torch.cat([visual, meta], dim=1))  # (B, 512)

        return {
            'y':       self.reg_head(fused),
            'log_var': self.log_var_head(fused),
            # auxiliary: reconstructed from visual features only
            'ndvi':    self.ndvi_head(visual).squeeze(1),
            'height':  self.height_head(visual).squeeze(1),
            'state':   self.state_head(visual),
            'month':   self.month_head(visual),
        }

## 7. Loss — learned uncertainty weighting

Instead of fixed weights, we use **Kendall & Gal (2017)** multi-task uncertainty:

$$\mathcal{L}_{\text{total}} = \sum_i \frac{1}{2\sigma_i^2} \mathcal{L}_i + \log \sigma_i$$

The network predicts $\log \sigma^2$ per target, so tasks with high inherent noise
automatically receive lower weight. This replaces the hand-tuned `w_num=0.30` in the baseline.

In [10]:
class CharmsLoss(nn.Module):
    def __init__(self, aux_weight=0.25):
        super().__init__()
        self.aux_weight = aux_weight
        self.mse = nn.MSELoss()
        self.ce  = nn.CrossEntropyLoss()

    def forward(self, out, y, x_num, x_cat):
        # ── Primary: uncertainty-weighted Smooth-L1 ──────────────────────────
        log_var = out['log_var']                     # (B, 5)
        precision = torch.exp(-log_var)              # 1/σ²
        residual  = F.smooth_l1_loss(out['y'], y, reduction='none')  # (B, 5)
        loss_main = (precision * residual + log_var).mean()

        # ── Auxiliary: metadata reconstruction ──────────────────────────────
        loss_ndvi   = self.mse(out['ndvi'],   x_num[:, 0])
        loss_height = self.mse(out['height'], x_num[:, 1])
        loss_state  = self.ce(out['state'],   x_cat[:, 0])
        loss_month  = self.ce(out['month'],   x_cat[:, 1])
        loss_aux = loss_ndvi + loss_height + loss_state + loss_month

        loss = loss_main + self.aux_weight * loss_aux

        return loss, {
            'loss': loss.item(), 'loss_main': loss_main.item(),
            'loss_aux': loss_aux.item(),
        }

## 8. Metrics

In [11]:
def r2_per_target(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2, axis=0)
    ss_tot = np.sum((y_true - y_true.mean(axis=0, keepdims=True)) ** 2, axis=0)
    return 1 - ss_res / np.clip(ss_tot, 1e-12, None)

def weighted_global_r2(y_true, y_pred):
    return float(np.mean(r2_per_target(y_true, y_pred)))

## 9. Train / validation loops

In [12]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running = {}
    for batch in loader:
        images  = batch['image'].to(device)
        targets = batch['targets'].to(device)
        x_num   = batch['x_num'].to(device)
        x_cat   = batch['x_cat'].to(device)

        optimizer.zero_grad()
        out = model(images, x_num, x_cat)
        loss, logs = criterion(out, targets, x_num, x_cat)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        for k, v in logs.items():
            running[k] = running.get(k, 0.0) + v
    for k in running:
        running[k] /= max(len(loader), 1)
    return running


@torch.no_grad()
def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    running, all_targets, all_preds = {}, [], []
    for batch in loader:
        images  = batch['image'].to(device)
        targets = batch['targets'].to(device)
        x_num   = batch['x_num'].to(device)
        x_cat   = batch['x_cat'].to(device)

        out = model(images, x_num, x_cat)
        _, logs = criterion(out, targets, x_num, x_cat)
        for k, v in logs.items():
            running[k] = running.get(k, 0.0) + v
        all_targets.append(targets.cpu().numpy())
        all_preds.append(out['y'].cpu().numpy())

    for k in running:
        running[k] /= max(len(loader), 1)
    y_true = np.concatenate(all_targets, axis=0)
    y_pred = np.concatenate(all_preds, axis=0)
    running['weighted_r2'] = weighted_global_r2(y_true, y_pred)
    return running, y_true, y_pred

## 10. Cross-validation training loop

We use `GroupKFold` grouped on `sample_id` — images from the same physical sampling
event must not appear in both train and valid (prevents data leakage).

In [13]:
gkf = GroupKFold(n_splits=N_FOLDS)
oof_preds   = np.zeros((len(train_df), len(TARGETS)))
oof_targets = np.zeros((len(train_df), len(TARGETS)))

for fold, (train_idx, valid_idx) in enumerate(
        gkf.split(train_df, groups=train_df['sample_id'])):

    print(f'\n===== FOLD {fold+1}/{N_FOLDS} =====')
    df_tr = train_df.iloc[train_idx].reset_index(drop=True)
    df_va = train_df.iloc[valid_idx].reset_index(drop=True)

    train_ds = BiomassDataset(df_tr, IMAGE_FOLDER, transforms=train_tfms)
    valid_ds = BiomassDataset(df_va, IMAGE_FOLDER, transforms=valid_tfms)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    model     = BiomassCharms(num_states=encoders['num_states'],
                               num_months=encoders['num_months']).to(DEVICE)
    criterion = CharmsLoss(aux_weight=0.25)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=EPOCHS, eta_min=LR * 0.01
    )

    best_r2, best_path = -1e9, f'best_fold{fold}.pt'

    for epoch in range(EPOCHS):
        tr_logs = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        va_logs, y_true, y_pred = valid_one_epoch(model, valid_loader, criterion, DEVICE)
        scheduler.step()

        print(f'  Epoch {epoch+1:02d} | '
              f'train_loss={tr_logs["loss"]:.4f} | '
              f'valid_loss={va_logs["loss"]:.4f} | '
              f'valid_r2={va_logs["weighted_r2"]:.4f}')

        if va_logs['weighted_r2'] > best_r2:
            best_r2 = va_logs['weighted_r2']
            torch.save(model.state_dict(), best_path)

    # Store OOF predictions
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    _, y_true_oof, y_pred_oof = valid_one_epoch(model, valid_loader, criterion, DEVICE)
    oof_preds[valid_idx]   = y_pred_oof
    oof_targets[valid_idx] = y_true_oof
    print(f'  Fold {fold+1} best R²: {best_r2:.4f}')

print(f'\nOOF weighted R² (log-space): {weighted_global_r2(oof_targets, oof_preds):.4f}')


===== FOLD 1/5 =====
  Epoch 01 | train_loss=2.9093 | valid_loss=2.3652 | valid_r2=-6.9161
  Epoch 02 | train_loss=1.9370 | valid_loss=1.5259 | valid_r2=-1.2171
  Epoch 03 | train_loss=1.1755 | valid_loss=0.9086 | valid_r2=-0.0648
  Epoch 04 | train_loss=0.7170 | valid_loss=0.5347 | valid_r2=0.2498
  Epoch 05 | train_loss=0.6761 | valid_loss=0.5494 | valid_r2=0.1791


C:\Users\somme\AppData\Local\Temp\ipykernel_32884\2048856777.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_location=DE

  Fold 1 best R²: 0.2498

===== FOLD 2/5 =====
  Epoch 01 | train_loss=2.9490 | valid_loss=2.7948 | valid_r2=-5.3203
  Epoch 02 | train_loss=2.0916 | valid_loss=1.8675 | valid_r2=-1.9197
  Epoch 03 | train_loss=1.3239 | valid_loss=1.1994 | valid_r2=-0.1033
  Epoch 04 | train_loss=0.8733 | valid_loss=0.9037 | valid_r2=0.0627
  Epoch 05 | train_loss=0.7220 | valid_loss=0.8775 | valid_r2=0.0771


C:\Users\somme\AppData\Local\Temp\ipykernel_32884\2048856777.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_location=DE

  Fold 2 best R²: 0.0771

===== FOLD 3/5 =====
  Epoch 01 | train_loss=2.9157 | valid_loss=2.4571 | valid_r2=-7.9371
  Epoch 02 | train_loss=1.9726 | valid_loss=1.5906 | valid_r2=-1.2653
  Epoch 03 | train_loss=1.1110 | valid_loss=0.9723 | valid_r2=-0.3775
  Epoch 04 | train_loss=0.7089 | valid_loss=0.5783 | valid_r2=0.1464
  Epoch 05 | train_loss=0.5655 | valid_loss=0.5002 | valid_r2=0.1748


...

  Fold 3 best R²: 0.1748

===== FOLD 4/5 =====
  Epoch 01 | train_loss=2.9521 | valid_loss=2.4185 | valid_r2=-7.9199
  Epoch 02 | train_loss=1.9925 | valid_loss=1.4368 | valid_r2=-1.0935
  Epoch 03 | train_loss=1.2130 | valid_loss=0.9004 | valid_r2=-0.0198
  Epoch 04 | train_loss=0.7800 | valid_loss=0.5747 | valid_r2=0.2328
  Epoch 05 | train_loss=0.6269 | valid_loss=0.4829 | valid_r2=0.2833


C:\Users\somme\AppData\Local\Temp\ipykernel_32884\2048856777.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_location=DE

  Fold 4 best R²: 0.2833

===== FOLD 5/5 =====
  Epoch 01 | train_loss=3.0835 | valid_loss=2.6340 | valid_r2=-7.4463
  Epoch 02 | train_loss=2.0907 | valid_loss=1.5983 | valid_r2=-1.4758
  Epoch 03 | train_loss=1.2457 | valid_loss=0.7149 | valid_r2=0.3060
  Epoch 04 | train_loss=0.8066 | valid_loss=0.4590 | valid_r2=0.3786
  Epoch 05 | train_loss=0.6525 | valid_loss=0.4236 | valid_r2=0.3640


C:\Users\somme\AppData\Local\Temp\ipykernel_32884\2048856777.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_location=DE

  Fold 5 best R²: 0.3786

OOF weighted R² (log-space): 0.2444


## 11. Test inference with TTA

**Test-Time Augmentation (TTA):** at inference we run each test image through 4 variants
(original + H-flip + V-flip + both flips), then average predictions.
This costs 4× inference time but consistently improves R² by ~0.01–0.02.

Note: metadata is **zeroed out** at test time since it is unavailable — this is the
exact CHARMS inference scenario.

In [14]:
@torch.no_grad()
def predict_test_tta(model, loader, device):
    model.eval()
    all_preds, all_paths = [], []

    for batch in loader:
        imgs  = batch['image'].to(device)
        x_num = batch['x_num'].to(device)
        x_cat = batch['x_cat'].to(device)

        # Zero out numeric metadata — unavailable at test time
        x_num_zero = torch.zeros_like(x_num)

        preds = [
            model(imgs,                                  x_num_zero, x_cat)['y'],
            model(imgs.flip(-1),                         x_num_zero, x_cat)['y'],  # H-flip
            model(imgs.flip(-2),                         x_num_zero, x_cat)['y'],  # V-flip
            model(imgs.flip(-1).flip(-2),                x_num_zero, x_cat)['y'],  # Both
        ]
        avg = torch.stack(preds, dim=0).mean(dim=0)     # (B, 5)
        all_preds.append(avg.cpu().numpy())
        all_paths.extend(batch['image_path'])

    return all_paths, np.concatenate(all_preds, axis=0)


# Average predictions across all folds
test_ds     = BiomassDataset(test_df, IMAGE_FOLDER, transforms=valid_tfms, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

fold_test_preds = []
for fold in range(N_FOLDS):
    m = BiomassCharms(num_states=encoders['num_states'],
                      num_months=encoders['num_months']).to(DEVICE)
    m.load_state_dict(torch.load(f'best_fold{fold}.pt', map_location=DEVICE))
    paths, preds = predict_test_tta(m, test_loader, DEVICE)
    fold_test_preds.append(preds)

test_image_paths = paths
test_preds_log   = np.mean(fold_test_preds, axis=0)    # average in log-space

# Invert the log1p transform
test_preds = np.expm1(np.clip(test_preds_log, 0, None))
print('Test predictions shape:', test_preds.shape)

C:\Users\somme\AppData\Local\Temp\ipykernel_32884\4262949488.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m.load_state_dict(torch.load(f'best_fold{fold}.pt', map_loc

Test predictions shape: (1, 5)


## 12. Build submission

In [15]:
pred_df = pd.DataFrame({
    'image_path':  test_image_paths,
    'Dry_Green_g': test_preds[:, 0],
    'Dry_Dead_g':  test_preds[:, 1],
    'Dry_Clover_g':test_preds[:, 2],
    'GDM_g':       test_preds[:, 3],
    'Dry_Total_g': test_preds[:, 4],
})

pred_long = pred_df.melt(
    id_vars='image_path', value_vars=TARGETS,
    var_name='target_name', value_name='target_pred',
)

sample_sub = pd.read_csv(TEST_CSV)
submission = sample_sub.merge(pred_long, on=['image_path', 'target_name'], how='left')
submission['target'] = submission['target_pred']
submission = submission[['sample_id', 'target']]

print(submission.shape)
print('Missing:', submission['target'].isna().sum())
submission.to_csv('submission_charms_v2.csv', index=False)
print('Saved submission_charms_v2.csv')
submission.head(10)

(5, 2)
Missing: 0
Saved submission_charms_v2.csv


,sample_id,target
0,ID1001187975__Dry_Clover_g,0.233605
1,ID1001187975__Dry_Dead_g,8.862324
2,ID1001187975__Dry_Green_g,15.817245
3,ID1001187975__Dry_Total_g,30.369110
4,ID1001187975__GDM_g,19.883827
